# Model Improvement

In [1]:
import pandas as pd
import numpy as np
import joblib
import time
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (RandomForestClassifier,GradientBoostingClassifier)
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

In [2]:
X_train = joblib.load("../models/X_train_engineered_processed.pkl")
X_test = joblib.load("../models/X_test_engineered_processed.pkl")
y_train = joblib.load("../models/y_train.pkl")
y_test = joblib.load("../models/y_test.pkl")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 75)
X_test : (1409, 75)
y_train: (5634,)
y_test : (1409,)


#### Cross Validation

In [3]:
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

#### evaluation function

In [4]:
def evaluate_improved_model(model_name,model,X_test,y_test,tuning_time):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_test_binary = (y_test == "Yes").astype(int)
    result = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test,y_pred),
        "Precision": precision_score(y_test,y_pred,pos_label="Yes",zero_division=0),
        "Recall": recall_score(y_test,y_pred,pos_label="Yes",zero_division=0),
        "F1-Score": f1_score(y_test,y_pred,pos_label="Yes",zero_division=0),
        "ROC-AUC": roc_auc_score(y_test_binary,y_proba),
        "PR-AUC": average_precision_score(y_test_binary,y_proba),
        "Tuning Time (sec)": tuning_time
    }

    return result

In [5]:
improved_results = []
best_models = {}
best_parameters = {}

#### Logistic Regression

In [6]:
logistic_model = LogisticRegression(max_iter=1000,random_state=42)
logistic_params = {
    "C": [0.01, 0.1, 1, 10],
    "class_weight": [None, "balanced"],
    "solver": ["liblinear", "lbfgs"]
}

In [7]:
random_search_lr = RandomizedSearchCV(
    estimator=logistic_model,
    param_distributions=logistic_params,
    n_iter=8,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [8]:
start_time = time.time()
random_search_lr.fit(X_train,y_train)
tuning_time_lr = time.time() - start_time

Fitting 5 folds for each of 8 candidates, totalling 40 fits


C:\Users\Admin\.virtualenvs\ai_customer_intelligence-q_Nzp0-8\Lib\site-packages\sklearn\model_selection\_search.py:1237: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan]
  warnings.warn(


In [9]:
print("\n===== Logistic Regression =====")
print("Best Parameters:")
print(random_search_lr.best_params_)
print("Best CV F1:")
print(round(random_search_lr.best_score_, 4))


===== Logistic Regression =====
Best Parameters:
{'solver': 'liblinear', 'class_weight': None, 'C': 0.01}
Best CV F1:
nan


In [10]:
best_lr = random_search_lr.best_estimator_
best_models["Logistic Regression"] = best_lr
best_parameters["Logistic Regression"] = random_search_lr.best_params_

In [11]:
lr_result = evaluate_improved_model(
    "Logistic Regression - Improved",
    best_lr,
    X_test,
    y_test,
    tuning_time_lr
)
improved_results.append(lr_result)

In [12]:
improved_results

[{'Model': 'Logistic Regression - Improved',
  'Accuracy': 0.8041163946061036,
  'Precision': 0.675,
  'Recall': 0.5053475935828877,
  'F1-Score': 0.5779816513761468,
  'ROC-AUC': 0.8409387997623291,
  'PR-AUC': 0.6272970099288591,
  'Tuning Time (sec)': 16.546611309051514}]

#### Decision Tree

In [13]:
decision_tree_model = DecisionTreeClassifier(random_state=42)
decision_tree_params = {
    "max_depth": [3, 5, 7, 10, 15, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "class_weight": [None, "balanced"]
}

In [14]:
random_search_dt = RandomizedSearchCV(
    estimator=decision_tree_model,
    param_distributions=decision_tree_params,
    n_iter=12,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [15]:
start_time = time.time()
random_search_dt.fit(X_train,y_train)
tuning_time_dt = time.time() - start_time

Fitting 5 folds for each of 12 candidates, totalling 60 fits


C:\Users\Admin\.virtualenvs\ai_customer_intelligence-q_Nzp0-8\Lib\site-packages\sklearn\model_selection\_search.py:1237: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


In [16]:
print("\n===== Decision Tree =====")
print("Best Parameters:")
print(random_search_dt.best_params_)
print("Best CV F1:")
print(round(random_search_dt.best_score_, 4))


===== Decision Tree =====
Best Parameters:
{'min_samples_split': 5, 'min_samples_leaf': 10, 'max_depth': 7, 'class_weight': None}
Best CV F1:
nan


In [17]:
best_dt = random_search_dt.best_estimator_
best_models["Decision Tree"] = best_dt
best_parameters["Decision Tree"] = random_search_dt.best_params_

In [18]:
dt_result = evaluate_improved_model(
    "Decision Tree - Improved",
    best_dt,
    X_test,
    y_test,
    tuning_time_dt
)
improved_results.append(dt_result)

In [19]:
improved_results

[{'Model': 'Logistic Regression - Improved',
  'Accuracy': 0.8041163946061036,
  'Precision': 0.675,
  'Recall': 0.5053475935828877,
  'F1-Score': 0.5779816513761468,
  'ROC-AUC': 0.8409387997623291,
  'PR-AUC': 0.6272970099288591,
  'Tuning Time (sec)': 16.546611309051514},
 {'Model': 'Decision Tree - Improved',
  'Accuracy': 0.7792760823278921,
  'Precision': 0.5975232198142415,
  'Recall': 0.516042780748663,
  'F1-Score': 0.5538020086083214,
  'ROC-AUC': 0.8130150610969026,
  'PR-AUC': 0.6052797389078259,
  'Tuning Time (sec)': 3.4834933280944824}]

#### KNN

In [20]:
knn_model = KNeighborsClassifier()
knn_params = {
    "n_neighbors": [3, 5, 7, 9, 11, 15],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

In [21]:
random_search_knn = RandomizedSearchCV(
    estimator=knn_model,
    param_distributions=knn_params,
    n_iter=10,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [22]:
start_time = time.time()
random_search_knn.fit(X_train,y_train)
tuning_time_knn = time.time() - start_time

Fitting 5 folds for each of 10 candidates, totalling 50 fits


C:\Users\Admin\.virtualenvs\ai_customer_intelligence-q_Nzp0-8\Lib\site-packages\sklearn\model_selection\_search.py:1237: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


In [23]:
print("\n===== KNN =====")
print("Best Parameters:")
print(random_search_knn.best_params_)
print("Best CV F1:")
print(round(random_search_knn.best_score_, 4))


===== KNN =====
Best Parameters:
{'weights': 'uniform', 'n_neighbors': 11, 'metric': 'euclidean'}
Best CV F1:
nan


In [24]:
best_knn = random_search_knn.best_estimator_
best_models["KNN"] = best_knn
best_parameters["KNN"] = random_search_knn.best_params_

In [25]:
knn_result = evaluate_improved_model(
    "KNN - Improved",
    best_knn,
    X_test,
    y_test,
    tuning_time_knn
)
improved_results.append(knn_result)

In [26]:
improved_results

[{'Model': 'Logistic Regression - Improved',
  'Accuracy': 0.8041163946061036,
  'Precision': 0.675,
  'Recall': 0.5053475935828877,
  'F1-Score': 0.5779816513761468,
  'ROC-AUC': 0.8409387997623291,
  'PR-AUC': 0.6272970099288591,
  'Tuning Time (sec)': 16.546611309051514},
 {'Model': 'Decision Tree - Improved',
  'Accuracy': 0.7792760823278921,
  'Precision': 0.5975232198142415,
  'Recall': 0.516042780748663,
  'F1-Score': 0.5538020086083214,
  'ROC-AUC': 0.8130150610969026,
  'PR-AUC': 0.6052797389078259,
  'Tuning Time (sec)': 3.4834933280944824},
 {'Model': 'KNN - Improved',
  'Accuracy': 0.7792760823278921,
  'Precision': 0.5940298507462687,
  'Recall': 0.5320855614973262,
  'F1-Score': 0.5613540197461213,
  'ROC-AUC': 0.8182980702162288,
  'PR-AUC': 0.5703231428372811,
  'Tuning Time (sec)': 1.0936594009399414}]

#### Random Forest

In [27]:
random_forest_model = RandomForestClassifier(random_state=42,n_jobs=-1)
random_forest_params = {
    "n_estimators": [100, 150, 200, 250],
    "max_depth": [5, 8, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
    "class_weight": [None, "balanced"]
}

In [28]:
random_search_rf = RandomizedSearchCV(
    estimator=random_forest_model,
    param_distributions=random_forest_params,
    n_iter=15,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [29]:
start_time = time.time()
random_search_rf.fit(X_train,y_train)
tuning_time_rf = time.time() - start_time

Fitting 5 folds for each of 15 candidates, totalling 75 fits


C:\Users\Admin\.virtualenvs\ai_customer_intelligence-q_Nzp0-8\Lib\site-packages\sklearn\model_selection\_search.py:1237: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


In [30]:
print("\n===== Random Forest =====")
print("Best Parameters:")
print(random_search_rf.best_params_)
print("Best CV F1:")
print(round(random_search_rf.best_score_, 4))


===== Random Forest =====
Best Parameters:
{'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': None, 'class_weight': None}
Best CV F1:
nan


In [31]:
best_rf = random_search_rf.best_estimator_
best_models["Random Forest"] = best_rf
best_parameters["Random Forest"] = random_search_rf.best_params_

In [32]:
rf_result = evaluate_improved_model(
    "Random Forest - Improved",
    best_rf,
    X_test,
    y_test,
    tuning_time_rf
)
improved_results.append(rf_result)

In [33]:
improved_results

[{'Model': 'Logistic Regression - Improved',
  'Accuracy': 0.8041163946061036,
  'Precision': 0.675,
  'Recall': 0.5053475935828877,
  'F1-Score': 0.5779816513761468,
  'ROC-AUC': 0.8409387997623291,
  'PR-AUC': 0.6272970099288591,
  'Tuning Time (sec)': 16.546611309051514},
 {'Model': 'Decision Tree - Improved',
  'Accuracy': 0.7792760823278921,
  'Precision': 0.5975232198142415,
  'Recall': 0.516042780748663,
  'F1-Score': 0.5538020086083214,
  'ROC-AUC': 0.8130150610969026,
  'PR-AUC': 0.6052797389078259,
  'Tuning Time (sec)': 3.4834933280944824},
 {'Model': 'KNN - Improved',
  'Accuracy': 0.7792760823278921,
  'Precision': 0.5940298507462687,
  'Recall': 0.5320855614973262,
  'F1-Score': 0.5613540197461213,
  'ROC-AUC': 0.8182980702162288,
  'PR-AUC': 0.5703231428372811,
  'Tuning Time (sec)': 1.0936594009399414},
 {'Model': 'Random Forest - Improved',
  'Accuracy': 0.7892122072391767,
  'Precision': 0.6323024054982818,
  'Recall': 0.4919786096256685,
  'F1-Score': 0.5533834586466

#### Gradient Boosting

In [34]:
gradient_boosting_model = GradientBoostingClassifier(random_state=42)
gradient_boosting_params = {
    "n_estimators": [50, 75, 100, 150],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

In [35]:
random_search_gb = RandomizedSearchCV(
    estimator=gradient_boosting_model,
    param_distributions=gradient_boosting_params,
    n_iter=10,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [36]:
start_time = time.time()
random_search_gb.fit(X_train,y_train)
tuning_time_gb = time.time() - start_time

Fitting 5 folds for each of 10 candidates, totalling 50 fits


C:\Users\Admin\.virtualenvs\ai_customer_intelligence-q_Nzp0-8\Lib\site-packages\sklearn\model_selection\_search.py:1237: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


In [37]:
print("\n===== Gradient Boosting =====")
print("Best Parameters:")
print(random_search_gb.best_params_)
print("Best CV F1:")
print(round(random_search_gb.best_score_, 4))


===== Gradient Boosting =====
Best Parameters:
{'n_estimators': 50, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_depth': 2, 'learning_rate': 0.05}
Best CV F1:
nan


In [38]:
best_gb = random_search_gb.best_estimator_
best_models["Gradient Boosting"] = best_gb
best_parameters["Gradient Boosting"] = random_search_gb.best_params_

In [39]:
gb_result = evaluate_improved_model(
    "Gradient Boosting - Improved",
    best_gb,
    X_test,
    y_test,
    tuning_time_gb
)
improved_results.append(gb_result)

In [40]:
improved_results

[{'Model': 'Logistic Regression - Improved',
  'Accuracy': 0.8041163946061036,
  'Precision': 0.675,
  'Recall': 0.5053475935828877,
  'F1-Score': 0.5779816513761468,
  'ROC-AUC': 0.8409387997623291,
  'PR-AUC': 0.6272970099288591,
  'Tuning Time (sec)': 16.546611309051514},
 {'Model': 'Decision Tree - Improved',
  'Accuracy': 0.7792760823278921,
  'Precision': 0.5975232198142415,
  'Recall': 0.516042780748663,
  'F1-Score': 0.5538020086083214,
  'ROC-AUC': 0.8130150610969026,
  'PR-AUC': 0.6052797389078259,
  'Tuning Time (sec)': 3.4834933280944824},
 {'Model': 'KNN - Improved',
  'Accuracy': 0.7792760823278921,
  'Precision': 0.5940298507462687,
  'Recall': 0.5320855614973262,
  'F1-Score': 0.5613540197461213,
  'ROC-AUC': 0.8182980702162288,
  'PR-AUC': 0.5703231428372811,
  'Tuning Time (sec)': 1.0936594009399414},
 {'Model': 'Random Forest - Improved',
  'Accuracy': 0.7892122072391767,
  'Precision': 0.6323024054982818,
  'Recall': 0.4919786096256685,
  'F1-Score': 0.5533834586466

#### XGBoost

In [41]:
y_train_xgb = (y_train == "Yes").astype(int)
y_test_xgb = (y_test == "Yes").astype(int)

#### Calculate class imbalance

In [42]:
negative_count = (y_train_xgb == 0).sum()
positive_count = (y_train_xgb == 1).sum()
scale_pos_weight = (negative_count / positive_count)
print("Scale pos weight:", scale_pos_weight)

Scale pos weight: 2.768561872909699


In [43]:
xgb_model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

In [44]:
xgb_params = {
    "n_estimators": [100, 150, 200],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

In [45]:
random_search_xgb = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_params,
    n_iter=12,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [46]:
start_time = time.time()
random_search_xgb.fit(X_train,y_train_xgb)
tuning_time_xgb = time.time() - start_time

Fitting 5 folds for each of 12 candidates, totalling 60 fits


In [47]:
print("\n===== XGBoost =====")
print("Best Parameters:")
print(random_search_xgb.best_params_)
print("Best CV F1:")
print(round(random_search_xgb.best_score_, 4))


===== XGBoost =====
Best Parameters:
{'subsample': 1.0, 'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
Best CV F1:
0.6353


In [48]:
best_xgb = random_search_xgb.best_estimator_
best_models["XGBoost"] = best_xgb
best_parameters["XGBoost"] = random_search_xgb.best_params_

In [49]:
def evaluate_xgb_model(model_name,model,X_test,y_test,tuning_time):
    y_pred_numeric = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = np.where(y_pred_numeric == 1,"Yes","No")
    y_test_binary = (y_test == "Yes").astype(int)
    result = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test,y_pred),
        "Precision": precision_score(y_test,y_pred,pos_label="Yes",zero_division=0),
        "Recall": recall_score(y_test,y_pred,pos_label="Yes",zero_division=0),
        "F1-Score": f1_score(y_test,y_pred,pos_label="Yes",zero_division=0),
        "ROC-AUC": roc_auc_score(y_test_binary,y_proba),
        "PR-AUC": average_precision_score(y_test_binary,y_proba),
        "Tuning Time (sec)": tuning_time
    }
    return result

In [50]:
xgb_result = evaluate_xgb_model(
    "XGBoost - Improved",
    best_xgb,
    X_test,
    y_test,
    tuning_time_xgb
)

improved_results.append(xgb_result)

In [51]:
improved_results

[{'Model': 'Logistic Regression - Improved',
  'Accuracy': 0.8041163946061036,
  'Precision': 0.675,
  'Recall': 0.5053475935828877,
  'F1-Score': 0.5779816513761468,
  'ROC-AUC': 0.8409387997623291,
  'PR-AUC': 0.6272970099288591,
  'Tuning Time (sec)': 16.546611309051514},
 {'Model': 'Decision Tree - Improved',
  'Accuracy': 0.7792760823278921,
  'Precision': 0.5975232198142415,
  'Recall': 0.516042780748663,
  'F1-Score': 0.5538020086083214,
  'ROC-AUC': 0.8130150610969026,
  'PR-AUC': 0.6052797389078259,
  'Tuning Time (sec)': 3.4834933280944824},
 {'Model': 'KNN - Improved',
  'Accuracy': 0.7792760823278921,
  'Precision': 0.5940298507462687,
  'Recall': 0.5320855614973262,
  'F1-Score': 0.5613540197461213,
  'ROC-AUC': 0.8182980702162288,
  'PR-AUC': 0.5703231428372811,
  'Tuning Time (sec)': 1.0936594009399414},
 {'Model': 'Random Forest - Improved',
  'Accuracy': 0.7892122072391767,
  'Precision': 0.6323024054982818,
  'Recall': 0.4919786096256685,
  'F1-Score': 0.5533834586466

In [52]:
improved_results_df = pd.DataFrame(
    improved_results
)

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "ROC-AUC",
    "PR-AUC",
    "Tuning Time (sec)"
]

improved_results_df[metric_columns] = (
    improved_results_df[metric_columns]
    .round(4)
)

improved_results_df

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC,Tuning Time (sec)
0,Logistic Regression - Improved,0.8041,0.6750,0.5053,0.5780,0.8409,0.6273,16.5466
1,Decision Tree - Improved,0.7793,0.5975,0.5160,0.5538,0.8130,0.6053,3.4835
2,KNN - Improved,0.7793,0.5940,0.5321,0.5614,0.8183,0.5703,1.0937
3,Random Forest - Improved,0.7892,0.6323,0.4920,0.5534,0.8362,0.6351,60.9545
4,Gradient Boosting - Improved,0.7892,0.6711,0.4037,0.5042,0.8404,0.6494,47.4827
5,XGBoost - Improved,0.7495,0.5185,0.7861,0.6249,0.8441,0.6559,22.2280


In [60]:
results_df = pd.read_csv("../models/phase5_baseline_model_results.csv")
results_df

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC,Training Time (sec)
0,XGBoost,0.756565,0.527337,0.799465,0.635494,0.846594,0.660382,1.026624
1,Random Forest,0.757275,0.529630,0.764706,0.625821,0.838903,0.643058,1.699015
2,Logistic Regression,0.736693,0.502564,0.786096,0.613139,0.840378,0.629892,0.461660
3,Decision Tree,0.726757,0.490909,0.794118,0.606742,0.821822,0.614773,0.137753
4,Gradient Boosting,0.804826,0.670103,0.521390,0.586466,0.844385,0.656003,5.346065
5,KNN,0.767921,0.568513,0.521390,0.543933,0.791043,0.531229,0.027988


In [61]:
baseline_compare = results_df[
    [
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "ROC-AUC",
        "PR-AUC"
    ]
].copy()
baseline_compare["Model"] = (baseline_compare["Model"]+ " - Baseline")
improved_compare = improved_results_df[
    [
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "ROC-AUC",
        "PR-AUC"
    ]
].copy()
comparison_df = pd.concat([baseline_compare,improved_compare],ignore_index=True)
comparison_df.round(4)

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC
0,XGBoost - Baseline,0.7566,0.5273,0.7995,0.6355,0.8466,0.6604
1,Random Forest - Baseline,0.7573,0.5296,0.7647,0.6258,0.8389,0.6431
2,Logistic Regression - Baseline,0.7367,0.5026,0.7861,0.6131,0.8404,0.6299
3,Decision Tree - Baseline,0.7268,0.4909,0.7941,0.6067,0.8218,0.6148
4,Gradient Boosting - Baseline,0.8048,0.6701,0.5214,0.5865,0.8444,0.6560
5,KNN - Baseline,0.7679,0.5685,0.5214,0.5439,0.7910,0.5312
6,Logistic Regression - Improved,0.8041,0.6750,0.5053,0.5780,0.8409,0.6273
7,Decision Tree - Improved,0.7793,0.5975,0.5160,0.5538,0.8130,0.6053
8,KNN - Improved,0.7793,0.5940,0.5321,0.5614,0.8183,0.5703
9,Random Forest - Improved,0.7892,0.6323,0.4920,0.5534,0.8362,0.6351


In [62]:
improved_results_df.sort_values(by="F1-Score",ascending=False).reset_index(drop=True)

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,PR-AUC,Tuning Time (sec)
0,XGBoost - Improved,0.7495,0.5185,0.7861,0.6249,0.8441,0.6559,22.2280
1,Logistic Regression - Improved,0.8041,0.6750,0.5053,0.5780,0.8409,0.6273,16.5466
2,KNN - Improved,0.7793,0.5940,0.5321,0.5614,0.8183,0.5703,1.0937
3,Decision Tree - Improved,0.7793,0.5975,0.5160,0.5538,0.8130,0.6053,3.4835
4,Random Forest - Improved,0.7892,0.6323,0.4920,0.5534,0.8362,0.6351,60.9545
5,Gradient Boosting - Improved,0.7892,0.6711,0.4037,0.5042,0.8404,0.6494,47.4827


In [63]:
improved_results_df.to_csv("../models/phase6_improved_model_results.csv",index=False)

In [64]:
comparison_df.to_csv("../models/phase5_vs_phase6_comparison.csv",index=False)
print("Baseline vs improved comparison saved successfully.")

Baseline vs improved comparison saved successfully.
